In [ ]:
#installs

In [ ]:
#imports
import os
import duckdb

Geef de locatie van de "verbeterde" bestanden

In [ ]:
load_directory = os.path.normpath(r".\\saved_parquet_files")
if not os.path.exists(load_directory):
    # maak directory als deze nog niet bestaat
     os.mkdir(load_directory)

Geef de locatie waar bestanden van de use case moeten worden opgeslagen

In [ ]:
save_directory = os.path.normpath(r".\\use_case_schijncomplexen")
if not os.path.exists(save_directory):
    # maak directory als deze nog niet bestaat
     os.mkdir(save_directory)

save- en load functies

In [ ]:
def save_complete_duckdb_memory(save_directory):
    """saves entire in-memory duckdb database to the given save_directory, in the form of parquet files"""
    duckdb.sql(f"""EXPORT DATABASE '{save_directory}' (FORMAT parquet)""" )

def empty_duckdb_memory():
    """Empty the in-memory duckdb database. Make sure you save first!"""
    schemas = duckdb.sql("""
                SELECT schema_name
                FROM information_schema.schemata
                WHERE schema_name NOT IN ('information_schema', 'pg_catalog', 'temp', 'main')""").fetchdf()
    for schema in schemas['schema_name']:
        tables = duckdb.sql(f"""
                        SELECT table_name
                        FROM information_schema.tables
                        WHERE table_schema = '{schema}'
        """).fetchdf()
        for table in tables['table_name']:
            duckdb.sql(f"DROP TABLE IF EXISTS {schema}.{table}")
        duckdb.sql(f"DROP SCHEMA IF EXISTS {schema}")

def load_duckdbset_to_memory(load_directory):
    """Loads an entire saved set of parquet files from the included "schema.sql" and "load.sql" files."""

    file_name = 'schema.sql'
    with open(os.path.join(load_directory, file_name), 'r') as schema:
        duckdb.sql(schema.read())

    file_name = 'load.sql'
    with open(os.path.join(load_directory, file_name), 'r') as load:
        duckdb.sql(load.read())
    duckdb.sql("SHOW SCHEMAS;")

# Identificeer schijncomplexen

# Schijf suggesties voor:
- welke complexen monumenten zouden moeten zijn
- welke monumenten complexen zouden moeten zijn
- welke complexen verwijderd kunnen worden, omdat dit nu rijksmonumenten zijn.
- welke rijksmonumenten verwijderd zouden kunnen worden, omdat deze nu deel zijn van een complex.